# Notebook 01b: Significant Feature Extraction — Yachida CRC Dataset

Runs immediately after `01_preprocessing.ipynb`. Extracts the species, metabolites, and
sample names that pass differential-abundance significance thresholds in each Yachida
stage-vs-Healthy comparison, builds union/frequency summaries across comparisons, draws
annotated volcano plots and a significant-feature heatmap, and saves a filtered subset for
downstream notebooks.

**Thresholds:** `QValue < Q_THRESH` AND `|CliffsD| > CLIFFSD_THRESH`.

`MeanDiff` (from `utils.differential_abundance()`) is a **linear-space mean difference in
whatever space the input matrix is already in** — CLR (natural-log ratio) for species,
log10 for metabolites — **not** a Log2 fold change. The old exploratory notebook computed a
`Log2FC = log2((mean2+eps)/(mean1+eps))` from means that were already CLR/log-transformed
(and frequently negative) — mathematically invalid, and it produced NaNs. `utils.py` no
longer computes a Log2FC at all.

**Effect-size gate uses `CliffsD`, not `MeanDiff`.** An earlier version reused the old
notebook's numeric threshold (`1.0`) but pointed it at `MeanDiff`. `MeanDiff` in CLR/log10
space is a *much* smaller quantity than a Log2 fold change, so `|MeanDiff| > 1.0` silently
zeroed out **every** comparison even though real FDR-significant hits existed. `CliffsD` is
rank-based and bounded in `[-1, 1]` regardless of the input transform, so it has no
scale-dependence problem. `CLIFFSD_THRESH = 0.33` is the conventional "medium effect"
cutoff (Vargha–Delaney / Romano et al.).

## Comparisons

**Primary (3)**, from `utils.COMPARISONS["YACHIDA-CRC-2019"]` — computed in NB01 and read
from the pickle: `Healthy` vs each of `Stage_0`, `Stage_I_II`, `Stage_III_IV`.
MP is excluded from the study, so there is no adenoma arm.

**Sensitivity (1)**, computed *here* in Section 4b: `Healthy` vs `PostSurgery` (the HS
group — patients whose colorectal cancer was resected and who are currently cancer-free).

This comparison is run in **its own FDR family** and is never merged into the primary
tables. Pooling it would both inflate the primary multiple-testing burden and invite the
reader to treat a resected gut as if it were a disease stage. Its purpose is the opposite:
to measure *how different* a post-surgical gut is from a healthy one, and therefore how
wrong it would have been to pool HS into either Healthy (as a control) or Early_CRC (as the
publication pipeline did). A large number of hits here is evidence that the v2.1 decision
to quarantine HS was necessary.

**Volcano plots:** the x-axis is `MeanDiff`, but the effect-size gate is `|Cliff's δ|`, so
there are no vertical threshold lines on the MeanDiff axis.

**Reads:** `RESULTS_DIR/intermediate/preprocessed_data_v2.pkl`
(`da_results_spc`/`da_results_mtb`, `harmonized_meta`, `transformed_species`/`transformed_mtb`,
`polyamine_columns`, `polyamine_proxies`, all `['YACHIDA-CRC-2019']`) — written by
`01_preprocessing.ipynb` with `EXPLORATORY = True` (the default).

**Writes:**
- `RESULTS_DIR/tables/YACHIDA_v2_significant_species_per_comparison.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_metabolites_per_comparison.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_species_union.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_metabolites_union.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_samples.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_sensitivity_healthy_vs_postsurgery_species.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_sensitivity_healthy_vs_postsurgery_metabolites.csv`
- `RESULTS_DIR/intermediate/yachida_significant_subset_v2.pkl` — dict with keys
  `species_filtered`, `metabolites_filtered`, `samples_meta`, `sig_species_union`,
  `sig_metabolites_union`, `thresholds`, `sig_species_freq`, `sig_metabolites_freq`,
  `sensitivity_da`.

`samples_meta` carries `Age, BMI, Gender, Brinkman.Index, Alcohol, CRC.Stage.Numeric,
Stage.Group`, plus `Tumor.Location.Site` / `Tumor.Location.Category` as **descriptive**
metadata only. Tumour location is admissible as a covariate only within a CRC-only sample
set; `utils.build_confounder_matrix()` raises if that rule is broken, so carrying the column
here cannot leak it into a mixed-group model.

In [ ]:
import sys
sys.path.insert(0, '.')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import utils
import viz
from utils import RESULTS_DIR, POLYAMINE_KEGG, FDR_THRESHOLD

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
viz.apply_theme()   # one shared theme and font (DejaVu Sans, Type 42): see viz.py

(RESULTS_DIR / 'tables').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'intermediate').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'figures' / 'differential').mkdir(parents=True, exist_ok=True)

# -- Thresholds ---------------------------------------------------------------
# Q_THRESH reuses utils.FDR_THRESHOLD (single source of truth) instead of a
# locally hardcoded 0.05 that could drift out of sync with the rest of the pipeline.
Q_THRESH = FDR_THRESHOLD

# CLIFFSD_THRESH gates on utils.differential_abundance()'s CliffsD column -- a
# rank-based effect size bounded in [-1, 1], scale-invariant regardless of whether
# the input is CLR (species) or log10 (metabolites). This replaces an earlier,
# buggy version of this notebook that thresholded |MeanDiff| > 1.0 (reusing the old
# Log2FC convention's numeric value unchanged) -- MeanDiff in CLR/log10 space rarely
# exceeds ~0.7 even for strongly significant hits, so that threshold silently zeroed
# out every comparison. 0.33 is the conventional "medium effect" Cliff's delta cutoff.
CLIFFSD_THRESH = 0.33

DATASET = 'YACHIDA-CRC-2019'

print(f'Significance thresholds: QValue < {Q_THRESH}  AND  |CliffsD| > {CLIFFSD_THRESH}')
print(f'Dataset: {DATASET}')


In [ ]:
with open(RESULTS_DIR / 'intermediate' / 'preprocessed_data_v2.pkl', 'rb') as f:
    preprocessed = pickle.load(f)

da_results_spc_all = preprocessed.get('da_results_spc')
da_results_mtb_all = preprocessed.get('da_results_mtb')
if da_results_spc_all is None or da_results_mtb_all is None:
    raise RuntimeError(
        "preprocessed_data_v2.pkl has da_results_spc/da_results_mtb = None. "
        "These are only populated when 01_preprocessing.ipynb's Step 7 (exploratory "
        "differential abundance) actually ran -- rerun it with EXPLORATORY = True "
        "(the default) before this notebook."
    )

da_results_spc = da_results_spc_all[DATASET]
da_results_mtb = da_results_mtb_all[DATASET]
harmonized_meta = preprocessed['harmonized_meta'][DATASET]
transformed_species = preprocessed['transformed_species'][DATASET]
transformed_mtb = preprocessed['transformed_mtb'][DATASET]
polyamine_columns = preprocessed['polyamine_columns'].get(DATASET, {})
polyamine_proxies = preprocessed.get('polyamine_proxies', {}).get(DATASET, {})

# Defined once, unconditionally, right after load -- NOT inside either heatmap
# block's own if-guard. The old notebook defined `comparisons_list` only inside
# `if len(sig_species_union) > 0:` (Section 4 below) and then reused it inside a
# SEPARATE `if len(sig_metabolites_union) > 0:` block; if species had zero
# significant hits while metabolites had some, that raised NameError. Every
# cell below uses this single comparisons_list instead of re-deriving `.keys()`.
comparisons_list = list(da_results_spc.keys())

# The primary family must not contain the sensitivity group: Section 4b runs that
# separately, with its own FDR correction.
assert not any('PostSurgery' in c or c.endswith('_vs_HS') for c in comparisons_list), \
    f'sensitivity comparison leaked into the primary DA family: {comparisons_list}'
assert not any('_vs_MP' in c for c in comparisons_list), 'MP is excluded from the study'

print(f'Primary comparisons for {DATASET} ({len(comparisons_list)}):')
for label in comparisons_list:
    print(f'  {label}')
print(f'\nGroups present in this cohort: '
      f'{harmonized_meta["Stage.Group"].value_counts().to_dict()}')
print('Sensitivity comparison (Healthy vs PostSurgery) is computed in Section 4b, '
      'in its own FDR family.')

---
## 1. Extract Significant Features per Comparison

In [ ]:
def apply_thresholds(da_df, q_thresh=Q_THRESH, fc_thresh=CLIFFSD_THRESH):
    '''Return rows passing QValue < q_thresh AND |CliffsD| > fc_thresh.

    Always returns a DataFrame with the expected DA columns present (even when
    empty), so downstream code can unconditionally do e.g. `sig_df['MeanDiff']`
    without a KeyError. Guards against da_df being None, empty, or missing
    'MeanDiff' entirely -- utils.differential_abundance() returns a genuinely
    empty DataFrame() (no columns at all) when a comparison has too few
    complete-case samples in one arm, which naive `da_df['MeanDiff']` indexing
    would otherwise crash on.
    '''
    empty_cols = ['Feature', 'MeanGroup1', 'MeanGroup2', 'MeanDiff',
                  'Statistic', 'PValue', 'CliffsD', 'QValue', 'Significant']
    if da_df is None or da_df.empty or 'MeanDiff' not in da_df.columns:
        return pd.DataFrame(columns=empty_cols)
    mask = (da_df['QValue'] < q_thresh) & (da_df['CliffsD'].abs() > fc_thresh)
    return da_df[mask].copy()


sig_species_per_comp = {}
sig_metabolites_per_comp = {}
pa_col_set = set(polyamine_columns.values())

print(f"{'Comparison':<40} {'Sig species':>12} {'Sig metabolites':>16}")
print('-' * 72)

for label in comparisons_list:
    sig_spc = apply_thresholds(da_results_spc.get(label))
    sig_mtb = apply_thresholds(da_results_mtb.get(label))

    sig_spc['Comparison'] = label
    sig_mtb['Comparison'] = label
    sig_spc['Direction'] = np.where(sig_spc['MeanDiff'] > 0, 'Up in case', 'Down in case')
    sig_mtb['Direction'] = np.where(sig_mtb['MeanDiff'] > 0, 'Up in case', 'Down in case')

    # Polyamine flag: polyamine_columns (from utils.find_polyamine_columns() in
    # 01_preprocessing) already maps KEGG ID -> exact column name via an mtb.map
    # KEGG lookup or an exact "startswith(kegg_id + '_')" check, so this
    # membership test is already exact, never substring-based.
    sig_mtb['IsPolyamine'] = sig_mtb['Feature'].isin(pa_col_set)

    sig_species_per_comp[label] = sig_spc
    sig_metabolites_per_comp[label] = sig_mtb

    print(f'{label:<40} {len(sig_spc):>12} {len(sig_mtb):>16}')


---
## 2. Annotated Volcano Plots (significant hits labelled)

Significance highlighting comes directly from `sig_df` (Step 1's `apply_thresholds()` output for this comparison), not recomputed from `da_df` internally -- see the `annotated_volcano()` docstring below.

In [ ]:
def annotated_volcano(da_df, sig_df, title, fc_col='MeanDiff', pa_cols=None,
                       q_thresh=Q_THRESH, fc_thresh=CLIFFSD_THRESH,
                       top_n_label=10, save_path=None):
    '''Volcano plot with significant hits (per `sig_df`) coloured and top hits labelled.

    `sig_df` -- the pre-computed apply_thresholds() output for this exact
    comparison -- is what actually determines which points are drawn as
    "significant" here. The old version accepted `sig_df` as a parameter but
    then silently ignored it and recomputed
    `is_sig = (da_df['QValue'] < q_thresh) & (da_df['Log2FC'].abs() > fc_thresh)`
    from da_df using this function's own default thresholds -- dead, duplicated
    logic that could disagree with whatever thresholds Step 1 actually used.
    Now `is_sig` is derived from `sig_df['Feature']` membership, so this plot
    always reflects Step 1's real output, not an independent recomputation.
    '''
    if da_df is None or da_df.empty:
        print(f'No data for: {title}')
        return

    fig, ax = plt.subplots(figsize=(11, 7))

    neg_log_q = -np.log10(da_df['QValue'].clip(lower=1e-300))
    sig_features = set(sig_df['Feature']) if sig_df is not None and not sig_df.empty else set()
    is_sig = da_df['Feature'].isin(sig_features)
    up = is_sig & (da_df[fc_col] > 0)
    down = is_sig & (da_df[fc_col] < 0)
    is_pa = da_df['Feature'].isin(pa_cols) if pa_cols else pd.Series(False, index=da_df.index)

    ax.scatter(da_df.loc[~is_sig, fc_col], neg_log_q[~is_sig],
               color='#cccccc', alpha=0.4, s=12, zorder=1, label='Not significant')
    ax.scatter(da_df.loc[up, fc_col], neg_log_q[up],
               color='#d62728', alpha=0.75, s=30, zorder=2, label=f'Up in case (n={up.sum()})')
    ax.scatter(da_df.loc[down, fc_col], neg_log_q[down],
               color='#1f77b4', alpha=0.75, s=30, zorder=2, label=f'Down in case (n={down.sum()})')
    if is_pa.any() and is_sig.any():
        pa_sig = is_pa & is_sig
        ax.scatter(da_df.loc[pa_sig, fc_col], neg_log_q[pa_sig],
                   color='gold', edgecolors='black', linewidth=0.8,
                   s=80, zorder=4, label=f'Polyamine (n={pa_sig.sum()})')

    ax.axhline(-np.log10(q_thresh), ls='--', color='grey', alpha=0.6, lw=1)
    # No vertical lines: the effect-size gate is |Cliff's delta|, not |MeanDiff|,
    # so a +/-fc_thresh line on the MeanDiff axis would be misleading.

    top_hits = da_df[is_sig].nsmallest(top_n_label, 'QValue')
    for _, row in top_hits.iterrows():
        y_val = -np.log10(max(row['QValue'], 1e-300))
        label_text = row['Feature']
        if ';' in label_text:
            label_text = label_text.split(';s__')[-1] if ';s__' in label_text else label_text.split(';')[-1]
        ax.annotate(
            label_text[:35],
            xy=(row[fc_col], y_val),
            xytext=(5, 3), textcoords='offset points',
            fontsize=6.5, alpha=0.9,
            arrowprops=dict(arrowstyle='-', color='grey', lw=0.4),
        )

    ax.set_xlabel('Mean Difference (case \u2212 Healthy, transformed space)', fontsize=11)
    ax.set_ylabel('\u2212log\u2081\u2080(Q-value)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.9)

    ax.text(0.01, 0.99,
            f"Gate: q<{q_thresh} and |Cliff's \u03b4|>{fc_thresh} (x-axis = MeanDiff)",
            transform=ax.transAxes, fontsize=8, va='top', color='grey')

    plt.tight_layout()
    if save_path:
        viz.save_figure(fig, save_path)
    plt.show()
    plt.close()


fig_dir = RESULTS_DIR / 'figures' / 'differential'

for label in comparisons_list:
    print(f'\n-- {label} --------------------------------------')
    annotated_volcano(
        da_results_spc.get(label), sig_species_per_comp[label],
        title=f'Yachida \u2014 Species: {label}',
        pa_cols=None,
        save_path=fig_dir / f'YACHIDA_v2_{label}_species_annotated_volcano.png'
    )
    annotated_volcano(
        da_results_mtb.get(label), sig_metabolites_per_comp[label],
        title=f'Yachida \u2014 Metabolites: {label}',
        pa_cols=pa_col_set,
        save_path=fig_dir / f'YACHIDA_v2_{label}_metabolites_annotated_volcano.png'
    )


---
## 3. Union of Significant Features Across All Comparisons

In [ ]:
all_sig_spc_df = pd.concat(sig_species_per_comp.values(), ignore_index=True)
all_sig_mtb_df = pd.concat(sig_metabolites_per_comp.values(), ignore_index=True)

sig_species_union = sorted(all_sig_spc_df['Feature'].unique())
sig_metabolites_union = sorted(all_sig_mtb_df['Feature'].unique())

print(f'Unique significant SPECIES     across all comparisons: {len(sig_species_union)}')
print(f'Unique significant METABOLITES across all comparisons: {len(sig_metabolites_union)}')


In [ ]:
spc_freq = (
    all_sig_spc_df.groupby('Feature')
    .agg(
        N_comparisons=('Comparison', 'nunique'),
        Comparisons=('Comparison', lambda x: ' | '.join(sorted(x.unique()))),
        Mean_MeanDiff=('MeanDiff', 'mean'),
        Min_QValue=('QValue', 'min'),
        Directions=('Direction', lambda x: ' | '.join(sorted(x.unique()))),
    )
    .reset_index()
    .sort_values('N_comparisons', ascending=False)
)

print('Top significant SPECIES (ranked by number of comparisons where significant):')
display(spc_freq.head(30))


In [ ]:
mtb_freq = (
    all_sig_mtb_df.groupby('Feature')
    .agg(
        N_comparisons=('Comparison', 'nunique'),
        Comparisons=('Comparison', lambda x: ' | '.join(sorted(x.unique()))),
        Mean_MeanDiff=('MeanDiff', 'mean'),
        Min_QValue=('QValue', 'min'),
        IsPolyamine=('IsPolyamine', 'any'),
        Directions=('Direction', lambda x: ' | '.join(sorted(x.unique()))),
    )
    .reset_index()
    .sort_values(['N_comparisons', 'Min_QValue'], ascending=[False, True])
)

print('Top significant METABOLITES (ranked by number of comparisons):')
display(mtb_freq.head(30))

pa_sig_mtbs = mtb_freq[mtb_freq['IsPolyamine']]
if not pa_sig_mtbs.empty:
    print(f'\nPolyamine metabolites in the significant set ({len(pa_sig_mtbs)}):')
    display(pa_sig_mtbs)


---
## 4. Significant-Feature Heatmaps (Species & Metabolites)

`comparisons_list` was already defined once, unconditionally, right after loading the data above -- so both heatmap blocks below use the same list consistently, even if species and metabolites don't both have significant hits (fixing the old `comparisons_list` `NameError` scope bug).

In [ ]:
if len(sig_species_union) > 0:
    top_species_heatmap = spc_freq.head(40)['Feature'].tolist()

    pivot_data = []
    for feat in top_species_heatmap:
        row = {'Feature': feat}
        for label in comparisons_list:
            match = all_sig_spc_df[
                (all_sig_spc_df['Feature'] == feat) &
                (all_sig_spc_df['Comparison'] == label)
            ]
            row[label] = match['MeanDiff'].values[0] if not match.empty else np.nan
        pivot_data.append(row)

    pivot_df = pd.DataFrame(pivot_data).set_index('Feature')

    short_idx = []
    for name in pivot_df.index:
        if ';s__' in name:
            short_idx.append(name.split(';s__')[-1][:40])
        else:
            short_idx.append(name[:40])
    pivot_df.index = short_idx

    fig_h = max(10, len(top_species_heatmap) * 0.35)
    fig, ax = plt.subplots(figsize=(max(10, len(comparisons_list) * 2.5), fig_h))
    sns.heatmap(
        pivot_df,
        cmap='RdBu_r', center=0, annot=True, fmt='.2f',
        linewidths=0.4, linecolor='#eeeeee',
        mask=pivot_df.isna(),
        cbar_kws={'label': 'MeanDiff (case \u2212 Healthy, CLR space)'},
        ax=ax
    )
    ax.set_title(
        f'Yachida \u2014 MeanDiff of Significant Species Across Comparisons\n'
        f'(q<{Q_THRESH}, |CliffsD|>{CLIFFSD_THRESH}; blank = not significant in that comparison)',
        fontsize=11
    )
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.xticks(rotation=35, ha='right', fontsize=9)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    viz.save_figure(plt.gcf(), RESULTS_DIR / 'figures' / 'differential' / 'YACHIDA_v2_significant_species_heatmap.png')
    plt.show()
    plt.close()
else:
    print('No significant species in any comparison -- skipping species heatmap.')


In [ ]:
if len(sig_metabolites_union) > 0:
    top_mtb_heatmap = mtb_freq.head(40)['Feature'].tolist()

    pivot_data_m = []
    for feat in top_mtb_heatmap:
        row = {'Feature': feat}
        for label in comparisons_list:
            match = all_sig_mtb_df[
                (all_sig_mtb_df['Feature'] == feat) &
                (all_sig_mtb_df['Comparison'] == label)
            ]
            row[label] = match['MeanDiff'].values[0] if not match.empty else np.nan
        pivot_data_m.append(row)

    pivot_m = pd.DataFrame(pivot_data_m).set_index('Feature')

    # Shorten KEGG-prefixed metabolite names for display (e.g.
    # "C00134_Putrescine(1,4-Butanediamine)" -> "Putrescine(1,4-Butanediamine)").
    # This just strips the leading "<KEGG_ID>_" token from the raw column name --
    # it does NOT reassign a POLYAMINE_KEGG lookup name, so it is unaffected by
    # the exact-vs-substring KEGG matching bug fixed in kegg_label() below.
    short_m_idx = []
    for name in pivot_m.index:
        if '_' in name and name[:6].startswith('C'):
            short_m_idx.append(name.split('_', 1)[-1][:40])
        else:
            short_m_idx.append(name[:40])
    pivot_m.index = short_m_idx

    fig_h = max(10, len(top_mtb_heatmap) * 0.35)
    fig, ax = plt.subplots(figsize=(max(10, len(comparisons_list) * 2.5), fig_h))
    sns.heatmap(
        pivot_m,
        cmap='RdBu_r', center=0, annot=True, fmt='.2f',
        linewidths=0.4, linecolor='#eeeeee',
        mask=pivot_m.isna(),
        cbar_kws={'label': 'MeanDiff (case \u2212 Healthy, log10 space)'},
        ax=ax
    )
    ax.set_title(
        f'Yachida \u2014 MeanDiff of Significant Metabolites Across Comparisons\n'
        f'(q<{Q_THRESH}, |CliffsD|>{CLIFFSD_THRESH})',
        fontsize=11
    )
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.xticks(rotation=35, ha='right', fontsize=9)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    viz.save_figure(plt.gcf(), RESULTS_DIR / 'figures' / 'differential' / 'YACHIDA_v2_significant_metabolites_heatmap.png')
    plt.show()
    plt.close()
else:
    print('No significant metabolites in any comparison -- skipping metabolite heatmap.')


---
## 4b. Sensitivity analysis: Healthy vs PostSurgery (HS)

The one place the `PostSurgery` group is used. It answers a question the group policy
depends on: **is a resected, currently cancer-free gut distinguishable from a healthy
one?**

- If yes, pooling HS into `Healthy` would have contaminated the control group, and pooling
  it into `Early_CRC` (what the publication pipeline did) would have put non-tumour samples
  into a tumour group. Either way the v2.1 decision to quarantine HS is justified.
- If no, HS looks like a control and the cost of excluding it is only lost sample size.

This runs in **its own BH family**, computed here rather than in NB01, so the primary
`Healthy vs Stage_*` q-values are completely unaffected by it. Nothing downstream consumes
these tables as evidence; they are reported, not propagated.

A caveat that limits interpretation either way: resection status is confounded with having
*had* cancer, and these 30 patients are post-treatment. This comparison cannot separate
"effect of surgery" from "residual effect of prior disease", and it is not used to claim
either.

In [ ]:
# Sensitivity DA: Healthy vs PostSurgery, in its OWN FDR family.
sensitivity_da = {}
n_ps = int((harmonized_meta['Stage.Group'] == 'PostSurgery').sum())
print(f'PostSurgery samples available: {n_ps}')

if n_ps < 10:
    print('Fewer than 10 PostSurgery samples -- sensitivity analysis skipped.')
else:
    for dtype, mat in [('species', transformed_species), ('metabolites', transformed_mtb)]:
        da = utils.differential_abundance(
            mat, harmonized_meta, 'Healthy', 'PostSurgery', group_col='Stage.Group')
        sensitivity_da[dtype] = da
        if da.empty:
            print(f'  {dtype}: no testable features')
            continue

        n_sig_q = int(da['Significant'].sum())
        gated = da[(da['QValue'] < Q_THRESH) & (da['CliffsD'].abs() > CLIFFSD_THRESH)]
        print(f'  {dtype}: {len(da)} tested | {n_sig_q} at q<{Q_THRESH} | '
              f'{len(gated)} also |CliffsD|>{CLIFFSD_THRESH}')

        out = (RESULTS_DIR / 'tables' /
               f'YACHIDA_v2_sensitivity_healthy_vs_postsurgery_{dtype}.csv')
        da.to_csv(out, index=False)
        print(f'    saved: {out.name}')

        if not gated.empty:
            top = gated.reindex(gated['CliffsD'].abs().sort_values(ascending=False).index).head(10)
            print(f'    top differing {dtype} (PostSurgery vs Healthy):')
            for _, r in top.iterrows():
                direction = 'higher' if r['MeanDiff'] > 0 else 'lower'
                print(f"      {str(r['Feature'])[:45]:45s} {direction} in PostSurgery  "
                      f"(delta={r['CliffsD']:+.2f}, q={r['QValue']:.3g})")

# Polyamine-specific view: are the study's target compounds themselves shifted?
if sensitivity_da.get('metabolites') is not None and not sensitivity_da['metabolites'].empty:
    pa_rows = sensitivity_da['metabolites'][
        sensitivity_da['metabolites']['Feature'].isin(pa_col_set)].copy()
    if not pa_rows.empty:
        pa_rows['Polyamine'] = pa_rows['Feature'].map(
            {v: POLYAMINE_KEGG[k] for k, v in polyamine_columns.items()})
        print('\nPolyamines in the Healthy-vs-PostSurgery sensitivity comparison:')
        print(pa_rows[['Polyamine', 'MeanDiff', 'CliffsD', 'QValue', 'Significant']]
              .round(4).to_string(index=False))

# Interpretation, stated from the numbers rather than asserted in prose.
_n_sig_total = sum(int(d['Significant'].sum()) for d in sensitivity_da.values()
                   if d is not None and not d.empty)
print(f'\nTotal FDR-significant features, Healthy vs PostSurgery: {_n_sig_total}')
if _n_sig_total > 0:
    print('=> A post-surgical gut IS measurably different from a healthy one, so pooling HS\n'
          '   into Healthy (control contamination) or into Early_CRC (non-tumour samples in a\n'
          '   tumour group) would both have biased the primary comparisons. Quarantining it\n'
          '   as its own group is the defensible choice.')
else:
    print('=> No detectable difference at this threshold. HS resembles Healthy here, so\n'
          '   excluding it from the primary set costs sample size but introduces no bias.\n'
          '   Note this is a 30-vs-127 comparison, so power is limited; absence of evidence\n'
          '   is not evidence of equivalence.')

---
## 5. Extract Relevant Sample Names

In [ ]:
groups_with_sig_hits = set(['Healthy'])  # always include reference
for label in comparisons_list:
    if not sig_species_per_comp[label].empty or not sig_metabolites_per_comp[label].empty:
        # label format: 'Healthy_vs_Stage_I_II'
        parts = label.replace('_vs_', '|').split('|')
        if len(parts) == 2:
            groups_with_sig_hits.add(parts[1].strip())

print(f'Study groups contributing significant features: {sorted(groups_with_sig_hits)}')

# Full Yachida confounder set carried through to downstream notebooks -- the old
# version only forwarded ['Sample', 'Study.Group', 'CRC.Stage.Numeric', 'Age',
# 'Gender', 'BMI'], silently dropping Brinkman.Index, Alcohol and tumour site.
# CRC.Stage.Numeric / Stage.Group come straight from utils.harmonize_metadata()'s
# output (the corrected utils.CRC_STAGE_NUMERIC / utils.STAGE_GROUP_MAP encodings)
# -- no local stage map is hand-rolled here.
#
# Tumour location travels as DESCRIPTIVE metadata only. It is admissible as a
# covariate exclusively within a CRC-only sample set, and
# utils.build_confounder_matrix(include_tumor_location=True) raises if that rule is
# violated -- so carrying the columns here cannot leak them into a mixed-group model.
CONFOUNDER_COLS = [
    'Sample', 'Study.Group',
    'Age', 'BMI', 'Gender', 'Brinkman.Index', 'Alcohol',
    'Tumor.Location.Site', 'Tumor.Location.Category',
    'CRC.Stage.Numeric', 'Stage.Group',
]

# Restrict the downstream subset to the PRIMARY analysis set. PostSurgery is used
# only in Section 4b above; letting it into this pickle would quietly put it into
# every notebook that reads yachida_significant_subset_v2.pkl.
primary = utils.primary_mask(harmonized_meta)
sig_samples = harmonized_meta[
    primary & harmonized_meta['Study.Group'].isin(groups_with_sig_hits)
][CONFOUNDER_COLS].copy()

assert not sig_samples['Stage.Group'].eq('PostSurgery').any(), \
    'PostSurgery must not enter the downstream subset'

print(f'\nTotal samples in relevant groups (primary only): {len(sig_samples)}')
print(sig_samples['Study.Group'].value_counts().to_string())
print(f"\nExcluded from this subset: "
      f"{int(utils.sensitivity_mask(harmonized_meta).sum())} PostSurgery sample(s)")

In [ ]:
# Sanity check: confirm sample IDs exist in both transformed matrices
sample_ids = set(sig_samples['Sample'])
in_species = set(transformed_species['Sample'])
in_mtb = set(transformed_mtb['Sample'])
in_all_three = sample_ids & in_species & in_mtb

print(f'Samples in metadata only:                 {len(sample_ids - in_species - in_mtb)}')
print(f'Samples present in all 3 (meta+spc+mtb):  {len(in_all_three)}')

sig_samples = sig_samples[sig_samples['Sample'].isin(in_all_three)].reset_index(drop=True)
print(f'Final sample set (complete data): {len(sig_samples)}')


---
## 6. Build Filtered DataFrames

In [ ]:
final_sample_ids = sig_samples['Sample'].tolist()

# Filter species matrix: only significant species columns + relevant samples
available_sig_spc = [s for s in sig_species_union if s in transformed_species.columns]
missing_spc = [s for s in sig_species_union if s not in transformed_species.columns]
if missing_spc:
    print(f'WARNING: {len(missing_spc)} significant species not found in transformed matrix '
          f'(may have been filtered out during 01_preprocessing Step 4):')
    for s in missing_spc[:10]:
        print(f'  {s}')

spc_filtered = (
    transformed_species[transformed_species['Sample'].isin(final_sample_ids)]
    [['Sample'] + available_sig_spc]
    .reset_index(drop=True)
)

# Filter metabolite matrix
available_sig_mtb = [m for m in sig_metabolites_union if m in transformed_mtb.columns]
missing_mtb = [m for m in sig_metabolites_union if m not in transformed_mtb.columns]
if missing_mtb:
    print(f'WARNING: {len(missing_mtb)} significant metabolites not found in transformed matrix.')

mtb_filtered = (
    transformed_mtb[transformed_mtb['Sample'].isin(final_sample_ids)]
    [['Sample'] + available_sig_mtb]
    .reset_index(drop=True)
)

print(f'\nFiltered species    matrix: {spc_filtered.shape[0]} samples x {len(available_sig_spc)} species')
print(f'Filtered metabolite matrix: {mtb_filtered.shape[0]} samples x {len(available_sig_mtb)} metabolites')
display(spc_filtered.head(3))
display(mtb_filtered.head(3))


---
## 7. Save All Outputs

Writes exactly these files (all `YACHIDA_v2_`-prefixed, matching the v2 pipeline's `_v2` file-naming convention in `CONTRACTS.md` -- not the bare `YACHIDA_...` names the old exploratory notebook used):
- `RESULTS_DIR/tables/YACHIDA_v2_significant_species_per_comparison.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_metabolites_per_comparison.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_species_union.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_metabolites_union.csv`
- `RESULTS_DIR/tables/YACHIDA_v2_significant_samples.csv`
- `RESULTS_DIR/intermediate/yachida_significant_subset_v2.pkl`

In [ ]:
tables_dir = RESULTS_DIR / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)

# 1. Per-comparison significant species
all_sig_spc_df.to_csv(
    tables_dir / 'YACHIDA_v2_significant_species_per_comparison.csv', index=False)

# 2. Per-comparison significant metabolites
all_sig_mtb_df.to_csv(
    tables_dir / 'YACHIDA_v2_significant_metabolites_per_comparison.csv', index=False)

# 3. Union species summary
spc_freq.to_csv(
    tables_dir / 'YACHIDA_v2_significant_species_union.csv', index=False)

# 4. Union metabolites summary
mtb_freq.to_csv(
    tables_dir / 'YACHIDA_v2_significant_metabolites_union.csv', index=False)

# 5. Significant samples
sig_samples.to_csv(
    tables_dir / 'YACHIDA_v2_significant_samples.csv', index=False)

# 6. Filtered matrices + full confounder metadata as a pkl for downstream use.
#    'sensitivity_da' is carried so NB08/NB09 can CITE the Healthy-vs-PostSurgery
#    result if needed, without ever re-running it or folding it into their own FDR.
subset = {
    'species_filtered': spc_filtered,
    'metabolites_filtered': mtb_filtered,
    'samples_meta': sig_samples,
    'sig_species_union': sig_species_union,
    'sig_metabolites_union': sig_metabolites_union,
    'thresholds': {'q': Q_THRESH, 'cliffsd': CLIFFSD_THRESH},
    'sig_species_freq': spc_freq,
    'sig_metabolites_freq': mtb_freq,
    'comparisons_primary': comparisons_list,
    'sensitivity_da': sensitivity_da,
}
with open(RESULTS_DIR / 'intermediate' / 'yachida_significant_subset_v2.pkl', 'wb') as f:
    pickle.dump(subset, f, protocol=pickle.HIGHEST_PROTOCOL)

print('All outputs saved:')
for fname in [
    'YACHIDA_v2_significant_species_per_comparison.csv',
    'YACHIDA_v2_significant_metabolites_per_comparison.csv',
    'YACHIDA_v2_significant_species_union.csv',
    'YACHIDA_v2_significant_metabolites_union.csv',
    'YACHIDA_v2_significant_samples.csv',
    'YACHIDA_v2_sensitivity_healthy_vs_postsurgery_species.csv     [sensitivity]',
    'YACHIDA_v2_sensitivity_healthy_vs_postsurgery_metabolites.csv [sensitivity]',
    'yachida_significant_subset_v2.pkl  [intermediate/]',
]:
    print(f'  {fname}')

In [ ]:
def kegg_label(col):
    '''Map a KEGG-prefixed metabolite column name (e.g.
    'C02714_N-Acetylputrescine') to its human-readable polyamine name by
    matching the LEADING KEGG ID exactly, not by loose substring containment.

    The old version did
    `next((v for k, v in POLYAMINE_KEGG.items() if k in p or v.lower() in p.lower()), p)`
    -- the `v.lower() in p.lower()` half matches any column whose name merely
    *contains* a polyamine name as a substring, so
    'C02714_N-Acetylputrescine' (N-Acetylputrescine) matched "putrescine" in
    POLYAMINE_KEGG and was mislabeled '(Putrescine)', even though C02714 is a
    chemically distinct KEGG compound from C00134 (Putrescine itself).
    '''
    kegg_id = str(col).split('_', 1)[0]
    return POLYAMINE_KEGG.get(kegg_id, col)


print('=' * 70)
print('YACHIDA SIGNIFICANT FEATURE EXTRACTION COMPLETE')
print(f'Thresholds: QValue < {Q_THRESH}  AND  |CliffsD| > {CLIFFSD_THRESH}')
print('=' * 70)
print(f'\nComparisons processed: {len(comparisons_list)}')
for label in comparisons_list:
    ns = len(sig_species_per_comp[label])
    nm = len(sig_metabolites_per_comp[label])
    print(f'  {label}: {ns} species, {nm} metabolites')

print(f'\nUnion significant species:     {len(sig_species_union)}')
print(f'Union significant metabolites: {len(sig_metabolites_union)}')
pa_in_sig = [m for m in sig_metabolites_union if m in pa_col_set]
print(f'  of which polyamines:         {len(pa_in_sig)}')
for p in pa_in_sig:
    print(f'    {p}  ({kegg_label(p)})')

print(f'\nRelevant samples extracted:    {len(sig_samples)}')
print(sig_samples['Study.Group'].value_counts().to_string())

print(f'\nFiltered species matrix:       {spc_filtered.shape}')
print(f'Filtered metabolites matrix:   {mtb_filtered.shape}')
